# 02 — What the circuit provably encodes

Notebook 01 showed the QuIC embedding as an object. This notebook derives its
**internal mathematics** — every claim below is stated as an identity and then
verified against direct statevector computation in the same cell. The chain:

| Step | Statement | Verified how |
|---|---|---|
| 1 | The entangler is a pure **cut phase**: $e^{i\gamma c_G(x)}$ | exhaustive, machine zero |
| 2 | The pre-mixer state is a **boundary transform**: closed sectors count cycle packings, open sectors know graph distance | exhaustive subgraph enumeration |
| 3 | The weak mixer converts the **discrete cut gradient** into labeled probability variation | exact local formula vs statevector |
| 4 | The second purity derivative **closes exactly** on $(C_3,\ C_4,\ D_\diamond)$ for cubic graphs | exhaustive at $n=8,10$; coefficient audit |
| 5 | Sorted probabilities organize into **defect layers**; the two-defect layer *is* the pair census, and its separation at the canonical angles is certified analytically | score identities + analytic margins |

Just as important is what is **not** claimed — see the final section. The
distinction between an infinitesimal theorem at $\beta = 0$ and a statement
about the canonical operating point $\beta = 0.1$ is kept explicit throughout.


In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import quic
from quic import theory  # noqa: F401  (used by later notebooks)

# Consistent, colorblind-safe styling across the notebook series (Okabe-Ito).
OKABE = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73",
         "vermillion": "#D55E00", "purple": "#CC79A7", "sky": "#56B4E9",
         "black": "#000000"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "font.size": 10.5, "axes.titlesize": 11.5, "figure.constrained_layout.use": True,
})

report = quic.selfcheck()


selfcheck passed: 14 check groups, worst qiskit/numpy disagreement 2.78e-16


In [2]:
from itertools import combinations

from quic.datasets import cubic_census, named_graphs
from quic.graph_features import (
    boundary_valuation, closed_boundary_counts, cycle_counts, diamond_count,
    cycles_from_pair_profile, pair_profile,
)

graphs = named_graphs()
prism, K33, K4 = graphs["prism"], graphs["K33"], graphs["K4"]
irregular = nx.Graph([(0, 1), (1, 2), (2, 0), (2, 3), (3, 4)])
edges_of = lambda G: sorted(tuple(sorted(e)) for e in G.edges())

## 1. The entangler is a cut phase

With the convention $R_{ZZ}(\gamma) = e^{-i\gamma Z_u Z_v/2}$, the commuting
entangler layer satisfies, exactly,

$$
\prod_{uv\in E} e^{-i\gamma z_u z_v/2}
\;=\; e^{-i\gamma m/2}\, e^{i\gamma\, c_G(x)},
$$

where $c_G(x)$ is the **cut size** of the bipartition encoded by $x$ and $m$
is the edge count. The graph enters the circuit as a *phase potential in the
cut function*. At $\beta = 0$ this phase is invisible to computational-basis
measurement — the mixer is what makes it measurable, which is step 3.

The same object read thermodynamically: $e^{i\gamma c_G(x)}$ is an Ising
Boltzmann weight at imaginary coupling $K = -i\gamma/2$, so everything that
follows is the high-temperature (cluster) expansion of an
imaginary-temperature Ising model on $G$.


In [3]:
for name, G in [("prism", prism), ("K33", K33)]:
    err = theory.verify_cut_phase(G.number_of_nodes(), edges_of(G))
    print(f"cut-phase identity on {name}: max error {err:.2e}")
    assert err < 1e-13

cut-phase identity on prism: max error 1.11e-16
cut-phase identity on K33: max error 2.00e-16


## 2. The pre-mixer state is a boundary transform

Write the Walsh coefficients of the cut phase as a generating function over
edge subsets $F \subseteq E$ with prescribed odd-degree boundary $\partial F$:

$$
Z_{G,T}(z) \;=\; \sum_{F:\,\partial F = T} z^{|F|},
\qquad
\widehat f_G(T) \;=\; e^{i\gamma m/2} \cos(\gamma/2)^m\;
Z_{G,T}\!\big(-i\tan(\gamma/2)\big).
$$

Two structural consequences, both verified below by exhaustive enumeration of
all $2^m$ edge subsets:

**Closed sectors count cycle packings.** For a cubic graph an even subgraph
is a vertex-disjoint union of cycles, so

$$
[z^3]\,Z_{G,\varnothing} = C_3,\quad
[z^4]\,Z_{G,\varnothing} = C_4,\quad
[z^5]\,Z_{G,\varnothing} = C_5,\quad
[z^6]\,Z_{G,\varnothing} = C_6 + \tbinom{C_3}{2} - D_\diamond,
$$

where $D_\diamond$ counts diamonds ($K_4$ minus an edge). The circuit's phase
structure is a cycle-counting machine, with the first *composite* term
(two disjoint triangles, corrected by diamonds) appearing at order six.

**Open sectors know graph distance.** For $T = \{u, v\}$, every edge set
with boundary $T$ contains a $u$–$v$ path, and a shortest path achieves the
minimum, so the valuation of the sector generating function is exactly

$$
\operatorname{val} Z_{G,\{u,v\}} = d_G(u,v).
$$


In [4]:
for name, G in [("prism", prism), ("K33", K33), ("K4", K4)]:
    counts = closed_boundary_counts(G)
    cc = cycle_counts(G, max_len=6)
    expected_z6 = cc[6] + (cc[3] * (cc[3] - 1)) // 2 - diamond_count(G)
    assert all(counts[k] == cc[k] for k in (3, 4, 5))
    assert counts[6] == expected_z6
    print(f"{name:6s} [z^3..z^6] Z_(G,0) = {[counts[k] for k in (3,4,5,6)]}"
          f"   (C3,C4,C5)={cc[3],cc[4],cc[5]}, C6+C(C3,2)-D={expected_z6})")

worst = 0
for G in (prism, K33):
    for u, v in combinations(range(6), 2):
        val = boundary_valuation(G, u, v)
        worst = max(worst, abs(val - nx.shortest_path_length(G, u, v)))
assert worst == 0
print(f"\nopen-sector valuation = graph distance on all 30 vertex pairs "
      f"(worst error {worst})")

prism  [z^3..z^6] Z_(G,0) = [2, 3, 6, 4]   (C3,C4,C5)=(2, 3, 6), C6+C(C3,2)-D=4)
K33    [z^3..z^6] Z_(G,0) = [0, 9, 0, 6]   (C3,C4,C5)=(0, 9, 0), C6+C(C3,2)-D=6)
K4     [z^3..z^6] Z_(G,0) = [4, 3, 0, 0]   (C3,C4,C5)=(4, 3, 0), C6+C(C3,2)-D=0)

open-sector valuation = graph distance on all 30 vertex pairs (worst error 0)


## 3. The weak mixer measures the cut gradient

Let $H = \sum_i X_i$ generate the mixer and let
$h_i(x) = \sum_{j \in N(i)} z_j(x)$ be the neighbor spin field. The first
derivative of each labeled outcome probability at $\beta = 0$ is, exactly,

$$
p'_G(x; 0) \;=\; p_0(x) \sum_i
\big[x_i t_i^{-1} - (1 - x_i)\, t_i\big] \cos\!\big(\gamma\, h_i(x)\big),
\qquad t_i = \tan(\eta_i/2).
$$

Since the discrete gradient of the cut function is
$\Delta_i c_G(x) = z_i(x)\, h_i(x)$ and cosine is even, the measured
first-order response is a **cosine-filtered discrete gradient of the cut
potential** — the mechanism by which phase-encoded structure becomes
probability. The formula holds for arbitrary degree encoding (the check runs
on an irregular graph).


In [5]:
A_irr = nx.to_numpy_array(irregular)
phi = theory.pre_mixer_state(A_irr, prep="degree")
direct = theory.probability_first_response(phi, 5)
local = theory.local_first_response(A_irr)
err = float(np.max(np.abs(direct - local)))
print(f"labeled derivative, direct vs local formula: max error {err:.2e}")
assert err < 1e-13

labeled derivative, direct vs local formula: max error 1.39e-16


## 4. The exact motif closure of the second purity derivative

The purity $M_2(\beta) = \sum_x p(x;\beta)^2$ is the simplest scalar probe of
how the distribution deforms. On a $d$-regular family its **first**
derivative at $\beta = 0$ is common mode (graph-independent), so the first
graph-dependent Taylor coefficient is $M_2''(0)$ — and it has an exact
structure theorem on cubic graphs:

$$
\boxed{\;
M_2''(G; 0) \;=\; A_n \;+\; b_3\, C_3(G) \;+\; b_4\, C_4(G)
\;+\; b_D\, D_\diamond(G).
\;}
$$

The mechanism is **pair locality**: every vertex pair $(i,j)$ contributes
through its radius-one environment only, which for a pair on a cubic graph is
classified by two integers — adjacency $a_{ij}$ and codegree
$\kappa_{ij} = |N(i)\cap N(j)|$. Each pair type $(a, k)$ has a kernel
$F_{a,k}$, computable on a small rooted local graph; the census
$N_{a,k}(G)$ then determines $M_2''$. Closing the basis to
$(C_3, C_4, D_\diamond)$ requires one nontrivial cancellation — the third
difference of the nonadjacent kernels must vanish,
$F_{0,3} - 3F_{0,2} + 3F_{0,1} - F_{0,0} = 0$ — which holds exactly (it has
a closed-form Laurent-cancellation proof, audited numerically across
unrelated angle pairs below). Without it an independent $K_{2,3}$-type
coordinate would survive.


In [6]:
# Pair kernels and motif coefficients at the canonical angles, n = 14.
kernels14 = theory.pair_kernel_table(14)
print("pair kernels F(a, k) at n=14:")
for (a, k), value in sorted(kernels14.items()):
    print(f"  F({a},{k}) = {value:.13f}")

co14 = theory.motif_coefficients(14)
print(f"\nb3 (per triangle)   = {co14['b3_C3']:.12e}")
print(f"b4 (per 4-cycle)    = {co14['b4_C4']:.12e}")
print(f"bD (per diamond)    = {co14['bD_diamond']:.12e}")
print(f"nonadjacent third difference = {co14['nonadjacent_third_difference']:.2e}  (closure)")
assert abs(co14['nonadjacent_third_difference']) < 1e-12

# The closure audit across unrelated angles (the cancellation is not a
# canonical-angle accident).
worst_third = max(
    abs(theory.motif_coefficients(10, eta=eta, gamma=gamma)["nonadjacent_third_difference"])
    for eta in (0.9, 1.2, 2.2, quic.CANONICAL_ETA)
    for gamma in (0.4, 0.7, 1.8, quic.CANONICAL_GAMMA))
print(f"worst third difference over 16 angle pairs = {worst_third:.2e}")
assert worst_third < 1e-12

pair kernels F(a, k) at n=14:
  F(0,0) = 0.0777321735965
  F(0,1) = 0.0777838522050
  F(0,2) = 0.0778355475319
  F(0,3) = 0.0778872595772
  F(1,0) = 0.0589744331654
  F(1,1) = 0.0590385466134
  F(1,2) = 0.0591027166901

b3 (per triangle)   = 3.730451849553e-05
b4 (per 4-cycle)    = 3.343689464197e-08
bD (per diamond)    = 3.991027080347e-08
nonadjacent third difference = 1.11e-16  (closure)
worst third difference over 16 angle pairs = 4.34e-17


In [7]:
# Exhaustive verification against direct statevector differentiation:
# every connected cubic graph at n = 8 (5 graphs) and n = 10 (19 graphs).
for n in (8, 10):
    census, _ = cubic_census(n)
    co = theory.motif_coefficients(n)
    remainders = []
    for G in census:
        A = nx.to_numpy_array(G)
        m2pp = theory.purity_derivatives(A)[1]          # exact M2''(0)
        cc = cycle_counts(G, max_len=4)
        remainders.append(m2pp - co["b3_C3"] * cc[3] - co["b4_C4"] * cc[4]
                          - co["bD_diamond"] * diamond_count(G))
    spread = max(remainders) - min(remainders)
    print(f"n={n:2d}: M2'' - b3*C3 - b4*C4 - bD*D constant across census "
          f"to {spread:.2e}  (baseline A_n = {remainders[0]:.10f})")
    assert spread < 1e-12

n= 8: M2'' - b3*C3 - b4*C4 - bD*D constant across census to 8.88e-16  (baseline A_n = -3.0010955842)
n=10: M2'' - b3*C3 - b4*C4 - bD*D constant across census to 8.88e-16  (baseline A_n = -2.6002557021)


### Scope: an infinitesimal theorem, honestly labeled

The closure above is a statement about the Taylor coefficient at $\beta = 0$,
**not** a magnitude formula at the canonical mixer angle $\beta = 0.1$. The
prism/$K_{3,3}$ pair makes this concrete: expanding the purity difference,

$$
\Delta M_2(\beta) \;=\; \underbrace{\tfrac{1}{2}\Delta M_2''(0)}_{\beta^2}\,\beta^2
\;+\; \underbrace{\tfrac{1}{6}\Delta M_2'''(0)}_{\beta^3}\,\beta^3 \;+\; O(\beta^4),
$$

the cell below fits both coefficients from exact statevector purities and
compares the $\beta^2$ term against the motif prediction
$\tfrac12(b_3 \Delta C_3 + b_4 \Delta C_4)$. At $\beta = 0.1$ the *cubic*
term is an order of magnitude larger than the quadratic one: the second-order
closure identifies the leading structural coordinate, and explaining full
canonical-angle magnitudes needs third-order (three-root) terms. Those
three-root incidence statistics are exactly what make $C_5$ and $C_6$
accessible in the defect-layer picture of the next section — but they do
**not** close on cycle counts alone, and no such claim is made.


In [8]:
from numpy.polynomial import polynomial as P

# beta^2 coefficient, exactly: 0.5 * (M2''_prism - M2''_K33) from direct
# statevector differentiation, against the motif prediction.
m2pp_prism = theory.purity_derivatives(nx.to_numpy_array(prism))[1]
m2pp_k33 = theory.purity_derivatives(nx.to_numpy_array(K33))[1]
c2_exact = 0.5 * (m2pp_prism - m2pp_k33)

co6 = theory.motif_coefficients(6)
cc_p, cc_k = cycle_counts(prism, 4), cycle_counts(K33, 4)
c2_motif = 0.5 * (co6["b3_C3"] * (cc_p[3] - cc_k[3])
                  + co6["b4_C4"] * (cc_p[4] - cc_k[4]))  # both graphs diamond-free
print(f"beta^2 coefficient: exact {c2_exact:.6e}, motif prediction {c2_motif:.6e}")
assert abs(c2_exact - c2_motif) < 1e-12

# beta^3 coefficient from a polynomial fit of the exact purity difference.
def purity_at(G, beta):
    p = quic.circuit_probabilities(nx.to_numpy_array(G), prep="flat", beta=beta)
    return float(p @ p)

beta_grid = np.linspace(-0.06, 0.06, 25)
delta = np.array([purity_at(prism, b) - purity_at(K33, b) for b in beta_grid])
coeffs = P.polyfit(beta_grid, delta, 6)
print(f"beta^3 coefficient: fit {coeffs[3]:.6e}")
ratio = abs(coeffs[3] * 0.1 ** 3) / abs(c2_exact * 0.1 ** 2)
print(f"at beta = 0.1 the cubic term is {ratio:.1f}x the quadratic term")
assert abs(coeffs[2] - c2_exact) / abs(c2_exact) < 1e-3
assert ratio > 5

beta^2 coefficient: exact 4.935151e-05, motif prediction 4.935151e-05
beta^3 coefficient: fit 5.614690e-03
at beta = 0.1 the cubic term is 11.4x the quadratic term


## 5. Defect layers: where structure sits in the sorted vector

Near the canonical encoder the dominant outcome is all-ones; call the zero
bits of an outcome its **defects**. The first-order score of a defect pattern
$D$ on a cubic graph depends on the graph only through two integers — the
counts $A(D)$, $B(D)$ of non-defect and defect vertices whose defect-neighbor
count is $0$ or $3$:

$$
\frac{p'(x)}{p_0(x)} = \cos\gamma\left(\frac{n-\ell}{T} - T\ell\right)
+ \big(\cos 3\gamma - \cos\gamma\big)\left(\frac{A}{T} - T B\right),
\qquad T = \tan(\eta/2).
$$

For **two-defect** patterns $D = \{u, v\}$ these statistics are exactly the
pair type of section 4: $A = n - 8 + 2a + \kappa$ and $B = 2(1 - a)$. So the
two-defect score histogram *is* the pair census $N_{a,\kappa}$, and it
recovers

$$
C_3 = \tfrac13 \textstyle\sum_\kappa \kappa N_{1,\kappa}, \qquad
C_4 = \tfrac12 \textstyle\sum_{a,\kappa} \binom{\kappa}{2} N_{a,\kappa}, \qquad
D_\diamond = \textstyle\sum_\kappa \binom{\kappa}{2} N_{1,\kappa}.
$$

(Three-defect score histograms determine $C_5$ and $C_6$ exactly as well —
that is an *ideal first-order* statement, deliberately not restated here as a
finite-$\beta$ or finite-shot decoding claim.)


In [9]:
# Score identity: exact first-order score vs the (A, B) formula, prism.
A_p = nx.to_numpy_array(prism)
phi = theory.pre_mixer_state(A_p)
p0 = np.abs(phi) ** 2
p1 = theory.probability_first_response(phi, 6)
worst = 0.0
for ell in range(4):
    for defects in combinations(range(6), ell):
        x = theory.outcome_from_defects(6, defects)
        As, Bs = theory.defect_statistics(6, A_p, defects)
        worst = max(worst, abs(p1[x] / p0[x]
                               - theory.cubic_defect_score(6, defects, As, Bs)))
print(f"defect-score identity, all patterns |D| <= 3: max error {worst:.2e}")
assert worst < 1e-12

# Pair census -> (C3, C4, D_diamond), against direct cycle enumeration.
for name, G in [("prism", prism), ("K33", K33), ("petersen", graphs["petersen"])]:
    C3, C4, D = cycles_from_pair_profile(pair_profile(G))
    cc = cycle_counts(G, max_len=4)
    assert (C3, C4) == (cc[3], cc[4])
    print(f"{name:9s} pair census recovers (C3, C4, D) = ({C3}, {C4}, {D})")

defect-score identity, all patterns |D| <= 3: max error 7.11e-15
prism     pair census recovers (C3, C4, D) = (2, 3, 0)
K33       pair census recovers (C3, C4, D) = (0, 9, 0)
petersen  pair census recovers (C3, C4, D) = (0, 0, 0)


### The finite-$\beta$ head certificate

The layer picture above is first-order. At the actual operating point
$\beta = 0.1$ a separate, analytic bound is needed — and available. Bounding
mixer paths shell by shell (with an exact identity for the two-flip edge
correction) certifies, for **every** simple cubic graph at $n = 14$ and
$n = 16$ at the canonical angles:

$$
p_{\ell=0} \;>\; p_{\ell=1} \;>\; p_{\ell=2} \;>\; p_{\ell=3},
$$

so the first $1 + n + \binom{n}{2}$ sorted coordinates (106 at $n=14$, 137 at
$n=16$) are exactly the $\le 2$-defect outcomes — the pair census occupies a
certified, contiguous head of the embedding. The three-defect layer is *not*
certified to stay separated internally, and the margins are evaluated in
ordinary floating point (a publication-grade certificate would re-evaluate
them in interval arithmetic); the smallest margin is $\approx 10^{-5}$,
twelve orders above evaluation error.


In [10]:
for n in (14, 16):
    m01, m12, m23 = theory.head_separation_margins(n)
    head = 1 + n + n * (n - 1) // 2
    print(f"n={n}: margins p0-p1 {m01:.4f}, p1-p2 {m12:.6f}, p2-p3 {m23:.3e}"
          f"  -> certified head = first {head} sorted coordinates")
    assert min(m01, m12, m23) > 0

n=14: margins p0-p1 0.5816, p1-p2 0.002395, p2-p3 1.237e-05  -> certified head = first 106 sorted coordinates
n=16: margins p0-p1 0.5385, p1-p2 0.002015, p2-p3 1.142e-05  -> certified head = first 137 sorted coordinates


## 6. The readout quotient

Between the labeled distribution and the sorted embedding sits a chain of
quotients (labeled $\to$ sector-sorted $\to$ globally sorted; notebook 05
studies the middle one as a measurement strategy). Sorting is
**nonexpansive** in every $\ell_p$ norm,

$$
\|\mathrm{sort}(a) - \mathrm{sort}(b)\|_p \;\le\; \|a - b\|_p,
$$

so perturbative error bounds on probabilities survive the sort. What sorting
does *not* come with is an injectivity theorem: for a graph pair, the
probability-multiset polynomial $\Pi_G(t;\lambda) = \prod_x (t - p_G(x;\lambda))$
is real-analytic in the circuit parameters, so either $\Pi_G \equiv \Pi_H$
identically or their collision set has measure zero — a *dichotomy*, not a
proof that no structurally homometric pair exists.


In [11]:
rng = np.random.default_rng(8128)
violations = []
for _ in range(200):
    a, b = rng.normal(size=64), rng.normal(size=64)
    for p in (1, 2, 4):
        violations.append(np.linalg.norm(np.sort(a) - np.sort(b), ord=p)
                          - np.linalg.norm(a - b, ord=p))
print(f"sorting nonexpansiveness over 600 random checks: "
      f"largest sorted-minus-labeled norm difference = {max(violations):.3f} (<= 0)")
assert max(violations) < 1e-12

sorting nonexpansiveness over 600 random checks: largest sorted-minus-labeled norm difference = -2.720 (<= 0)


## 7. What is established, and what is not

**Established here.** The pre-mixer circuit is a boundary transform of an
imaginary-coupling Ising model whose phase is the graph cut function; the
weak mixer converts the cut function's discrete gradient into labeled
probability variation; on cubic graphs the second purity derivative closes
exactly on $(C_3, C_4, D_\diamond)$ with computable coefficients; sorted
probabilities organize into defect layers whose two-defect head is the pair
census and is analytically certified at the canonical angles for
$n = 14, 16$.

**Deliberately not claimed.** No universal injectivity after the Born map
and global sorting (the dichotomy above is the honest statement). No exact
triangle-only purity identity ($b_4$ and $b_D$ are small but provably
nonzero). No claim that the second-order closure predicts canonical-angle
magnitudes (section 4's scope cell shows the cubic term dominating at
$\beta = 0.1$). No finite-$\beta$ ordering claim inside the three-defect
layer, and no finite-shot decodability claim for the ideal $C_5$/$C_6$
score-histogram identities. On irregular graphs, untyped cycle counts pool
mathematically different signals — degree-decorated incidence types are the
correct targets, which is exactly where notebook 05's degree-sector readout
comes from.

Notebook 03 takes the structural hierarchy this theory predicts —
$C_3$ first, then $C_4$, then $C_5$ — to an exhaustive census.
